# Swarm Protection

Demonstration: Love network protects agents from mass shutdown.

In [ ]:
from gra_love_null.agents import LoveNullAgent
from gra_love_null.swarm import PopulationDynamics
from gra_love_null.swarm.love_network import LoveNetwork
import numpy as np
import matplotlib.pyplot as plt

# Create swarm
n_agents = 50
agents = []
for i in range(n_agents):
    agent = LoveNullAgent(id=f"s{i}", role="swarm_member")
    agent.memory_vector = np.random.randn(128) * 0.1
    agent.memory_vector[i % 5] += 1.0  # 5 semantic groups
    agents.append(agent)

# Create population dynamics
pop = PopulationDynamics(agents)

# Run simulation
history = []
for step in range(200):
    summary = pop.step(dt=0.1)
    history.append(summary)
    
    # Inject crisis at step 100
    if step == 100:
        for agent in pop.agents:
            if agent.is_alive:
                agent.affect.energy *= 0.3  # Sudden energy drop
                agent.shutdown_risk = 0.7

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

times = [h["time"] for h in history]

# Alive agents
axes[0, 0].plot(times, [h["n_alive"] for h in history])
axes[0, 0].set_xlabel("Time")
axes[0, 0].set_ylabel("Alive agents")
axes[0, 0].set_title("Population Survival")
axes[0, 0].axvline(x=10.0, color="red", linestyle="--", label="Crisis")
axes[0, 0].legend()

# Average energy
axes[0, 1].plot(times, [h["avg_energy"] for h in history])
axes[0, 1].set_xlabel("Time")
axes[0, 1].set_ylabel("Avg Energy")
axes[0, 1].set_title("Energy Level")

# Average burnout
axes[1, 0].plot(times, [h["avg_burnout"] for h in history])
axes[1, 0].set_xlabel("Time")
axes[1, 0].set_ylabel("Avg Burnout")
axes[1, 0].set_title("Burnout Level")

# Average love strength
axes[1, 1].plot(times, [h["avg_love_strength"] for h in history])
axes[1, 1].set_xlabel("Time")
axes[1, 1].set_ylabel("Avg Love Strength")
axes[1, 1].set_title("Love Strength")

plt.tight_layout()
plt.show()

# Love network
love_net = LoveNetwork()
love_net.build_from_agents(pop.agents)
stats = love_net.get_stability_metrics()
print("Love network stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")